# Entraînement du classifieur mammo bénin/malin (CBIS-DDSM) — Colab GPU

Pipeline **de bout en bout** : dataset HF privé → manifestes (split par patient stratifié) →
entraînement (`cropped` **et** `full`) → comparaison → publication de l'artefact versionné sur HF.

**Cibles de fiabilité** (porte de validation) : **AUC ≥ 0,85**, **sensibilité ≥ 0,90**, spécificité ≥ 0,75.

**Pré-requis** : Runtime **GPU** (Exécution → Modifier le type d'exécution → T4) ; un **token HF**
(`read` pour le dataset, `write` pour publier) ; le dataset CBIS déjà poussé dans le repo HF privé
`Mailcoding/ifar-mammo-rois` (images + CSV de description de cas + `dicom_info.csv`).

> ⚠️ Le modèle qui part en production est le **cropped** (le `ml-service` classe des ROIs détectées
> par YOLO). Le **full** est entraîné pour comparaison. Ne publie/déploie **aucun** modèle tant que
> les cibles ne sont pas atteintes (cf. `GOVERNANCE.md` / porte de validation SaMD).

## 1. Paramètres

In [ ]:
HF_DATASET_REPO = 'Mailcoding/ifar-mammo-rois'   # repo dataset HF privé (images + CSV)
HF_MODEL_REPO   = 'Mailcoding/ifar-mammo-classifier'  # repo model HF privé (artefact publié)
VERSION         = 'v0.1.0'

# Chemins RELATIFS dans le repo dataset (adapte à ta structure) :
CASE_CSVS = [
    'mass_case_description_train_set.csv',
    'calc_case_description_train_set.csv',
    'mass_case_description_test_set.csv',
    'calc_case_description_test_set.csv',
]
DICOM_INFO = 'dicom_info.csv'   # export Kaggle (SeriesUID->jpeg). Mets None si arbre DICOM->jpeg.
VAL_FRAC   = 0.2
SEED       = 42
USES       = ['cropped', 'full']   # entraîne les deux variantes

## 2. Installer le package `ifar-mlops`

In [ ]:
# Option A — dépôt dédié (une fois peuplé) :
!git clone https://github.com/mailcoding/ifar-mlops.git 2>/dev/null && cd ifar-mlops || true
# Option B (repli) — monorepo produit : clone puis `%cd <repo>/mlops`.
%cd /content/ifar-mlops
!pip install -e '.[train]' -q
import torch; print('CUDA dispo :', torch.cuda.is_available())

## 3. Authentification Hugging Face

In [ ]:
import os
from huggingface_hub import login
os.environ['HF_TOKEN'] = 'hf_xxx'   # ← ton token (read dataset / write publication)
login(os.environ['HF_TOKEN'])

## 4. Télécharger le dataset (snapshot du repo HF privé)

In [ ]:
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download(repo_id=HF_DATASET_REPO, repo_type='dataset',
                             local_dir='/content/data')
print('Dataset local :', DATA_DIR)
!ls -la {DATA_DIR} | head

## 5. Générer les manifestes (split par patient stratifié) + class_weights

On utilise l'API `build()` pour récupérer les `class_weights` suggérés (déséquilibre) et les
injecter dans la config d'entraînement.

In [ ]:
import os, csv, json
from mlops.datasets.build_cbis_manifest import build

def write_manifest(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['path', 'label'])
        for r in rows: w.writerow([r['path'], r['label']])

case_paths = [os.path.join(DATA_DIR, c) for c in CASE_CSVS]
dicom_info = os.path.join(DATA_DIR, DICOM_INFO) if DICOM_INFO else None
REPORTS = {}
for use in USES:
    rep = build(case_paths, use=use, dicom_info=dicom_info, images_root=DATA_DIR,
                val_frac=VAL_FRAC, seed=SEED, verify=True)
    s = rep['stats']
    assert not s['patient_leak'], f'FUITE PATIENT ({use}) !'
    out = f'/content/manifests/{use}'
    write_manifest(rep['train'], f'{out}/train.csv')
    write_manifest(rep['val'],   f'{out}/val.csv')
    REPORTS[use] = rep
    print(f"[{use}] résolues={s['n_resolved']} (manquants {s['missing_files']}) | "
          f"train {s['train']} / val {s['val']} | class_weights {s['suggested_class_weights']}")

## 6. Entraîner les deux variantes (`cropped` puis `full`)

In [ ]:
import yaml, copy, subprocess
base = yaml.safe_load(open('configs/mammo_classifier.yaml'))
for use in USES:
    cfg = copy.deepcopy(base)
    cfg['data']['train_manifest'] = f'/content/manifests/{use}/train.csv'
    cfg['data']['val_manifest']   = f'/content/manifests/{use}/val.csv'
    cfg['train']['class_weights'] = REPORTS[use]['stats']['suggested_class_weights']
    cfg['export']['out_dir'] = f'/content/artifacts/{use}'
    cfg['export']['version'] = VERSION
    p = f'/content/config_{use}.yaml'; yaml.safe_dump(cfg, open(p, 'w'))
    print(f'\n===== ENTRAÎNEMENT [{use}] (class_weights={cfg["train"]["class_weights"]}) =====')
    subprocess.run(['python', '-m', 'mlops.train.train_mammo_classifier', '--config', p], check=True)

## 7. Comparer les métriques des deux variantes

In [ ]:
import json
for use in USES:
    m = json.load(open(f'/content/artifacts/{use}/manifest.json'))['metrics']
    print(f"[{use:7}] AUC={m.get('auc')} sens={m.get('sensitivity')} spec={m.get('specificity')} "
          f"| @sens>={m.get('sensitivity_target')}: thr={m.get('operating_threshold_sens')} "
          f"spec={m.get('specificity_at_target_sens')}")
print('\nRappel cibles : AUC>=0.85, sensibilité>=0.90, spécificité>=0.75.')

## 8. Publier l'artefact de production (variante `cropped`)

Ne publie **que** si les cibles sont atteintes. La variante `cropped` correspond à l'inférence
produit. (Pour archiver aussi le `full`, publie-le sous un autre tag / repo.)

In [ ]:
from mlops.registry import publish_artifact
PROD = 'cropped'
m = json.load(open(f'/content/artifacts/{PROD}/manifest.json'))['metrics']
ok = (m.get('auc') or 0) >= 0.85 and (m.get('sensitivity') or 0) >= 0.90
if not ok:
    print(f'⛔ Cibles non atteintes ({PROD}) — publication ANNULÉE. metrics={m}')
else:
    url = publish_artifact(f'/content/artifacts/{PROD}', HF_MODEL_REPO, VERSION)
    print('✅ Publié :', url, '(tag', VERSION, ')')

## 9. Suite (hors notebook)

- **Porte de validation clinique** (SaMD / IEC 62304 / ISO 14971) avant de marquer le modèle
  `validated: true` dans `manifest.json`.
- **Consommation produit** : le Space `ml-service` récupère la version publiée
  (`pull_artifact` / `deploy-ml-space.yml`) — le nom de fichier reste `efficientnet_b0.pth`.
- **Suivi prod** : dashboard ops `/metrics`.